# 3 · HDC proposes, the graph validates

Notebook 1 gave us an exact graph in LanceDB and lance-graph. Notebook 2 gave us three
operators on hypervectors: binding ⊗, bundling ⊕ and permutation ρ. Here we put them
together. We encode the **whole warehouse trial graph as one hypervector**, ask it questions,
and then check its answers against the graph.

> **HDC proposes candidates. The graph validates whether those paths actually exist.**

## Setup

We reuse the warehouse scenario's data and loaders from notebook 1 (`NODES`, `EDGES`,
`write_graph`, `graph_config`). See the code for this in `warehouse_trial.py`.
LanceDB is used to store both the graph (via `lance-graph`) and the hypervectors. And torch plus
[torchhd](https://github.com/hyperdimensional-computing/torchhd) for the HDC operators.

In [1]:
from warehouse_trial import NODES, EDGES, write_graph, graph_config  # import before lancedb: it quiets LanceDB's log

import lancedb
import torch
import torchhd                         # HDC operators: random, bind, multiset (bundle), permute
from lance_graph import CypherQuery    # Cypher over the LanceDB tables, as in notebook 1

## The graph, revisited

The graph has an important job in this workflow: it validates the answers HDC proposes to ensure that the paths actually exist in the data. So we
store it in LanceDB exactly as in notebook 1, with one table per node label and one per
relationship type. `write_graph` and `graph_config` are those same notebook 1 cells,
wrapped up as two functions.

In [2]:
db = lancedb.connect("warehouse.lancedb")   # a folder on disk; created if missing
write_graph(db)                             # Robot, Aisle, ..., VISITED, HAS_HAZARD, HANDLED_BY tables
cfg = graph_config()                        # which columns are node IDs and which are edge endpoints

# lance-graph reads Arrow tables, keyed by table name
datasets = {name: db.open_table(name).to_arrow() for name in db.list_tables().tables}

## What is a codebook?

HDC can't compute with the string `"R04"`. Every distinct thing we want to talk about first
gets its own **hypervector**: each of the 11 nodes, and each of the 3 relationship types.
The **codebook** is the lookup table from name to hypervector, like a Python dict. You can
think of it as node IDs that you can do arithmetic with.

### Random, and that's the point

The hypervectors carry no meaning. Each is just $D$ random entries of $+1$ or $-1$ (the MAP
model from notebook 2). That works because two random hypervectors are almost orthogonal,
$\cos(\mathbf{a}, \mathbf{b}) \approx 0 \pm 1/\sqrt{D}$, so every name gets a hypervector
that's clearly distinct from all the others, without anyone designing it.

### Fixed representations, shared by the whole fleet

Hypervectors are fixed representations generated by an encoder, and are shared
by the whole fleet. That matters because everything later is built from codebook entries with
the operators from notebook 2. The edge R04 → aisle-9, for example, is stored as
$\mathbf{R04} \otimes \mathbf{VISITED} \otimes \rho(\mathbf{aisle9})$. To read it back, you
unbind with those exact $\mathbf{R04}$ and $\mathbf{VISITED}$ hypervectors and compare the
result with that exact $\mathbf{aisle9}$. Use a different hypervector for any of them and all
you get back is noise.

We fix the representations by fixing the random seed. With seed 0, the same names in the same
order and $D = 10{,}000$, the next cell produces exactly the hypervectors from notebook 2.

In [3]:
torch.manual_seed(0)  # fixed seed → the same codebook every run, and the same as notebook 2
D = 10_000            # entries per hypervector; noise between random ones is about 1/√D = 0.01

# 11 node IDs, then 3 relationship types: one hypervector for each
names = [n["id"] for rows in NODES.values() for n in rows] + list(EDGES)
hv = torchhd.random(len(names), D, vsa="MAP")   # a 14 × 10,000 tensor of random ±1 entries
codebook = dict(zip(names, hv))                 # name → hypervector

print(len(codebook), "hypervectors of length", D)
print("R04 starts with", codebook["R04"][:8].tolist())
print(f"cos(R04, R11) = {torchhd.cosine_similarity(codebook['R04'], codebook['R11']).item():+.3f}   (unrelated, as expected)")

14 hypervectors of length 10000
R04 starts with [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
cos(R04, R11) = -0.003   (unrelated, as expected)


## The codebook in LanceDB: going from hypervector back to name

Going from name to hypervector is a dict lookup. The reverse is what we'll need: our
questions return a **noisy** hypervector, close to one codebook entry but not equal to it.
Finding the closest entry is a nearest-neighbour search, which is exactly what LanceDB's
vector search does. So we store each name with its `kind` and its hypervector.

In [4]:
# kind: the node label (Robot, Aisle, ...) or "relationship"; handy for reading results
kinds = {n["id"]: label for label, rows in NODES.items() for n in rows} | {r: "relationship" for r in EDGES}

table = db.create_table(
    "hypervectors",
    [{"name": n, "kind": kinds[n], "vector": codebook[n].tolist()} for n in names],
    mode="overwrite",
)
table.count_rows()

14

### Cleanup

`cleanup` asks LanceDB for the `k` codebook entries nearest to a query hypervector by cosine.
LanceDB reports cosine *distance*, $1 - \cos$, so we turn it back into a similarity score.
It takes the table as an argument, because later we'll build a second, smaller codebook. As a
sanity check, a clean codebook entry finds itself with a score of 1.0, and everything else
scores near 0.

In [5]:
def cleanup(query, table, k=4):
    hits = table.search(query.numpy()).metric("cosine").limit(k).to_list()
    return [(h["name"], round(1 - h["_distance"], 3)) for h in hits]  # distance → similarity

cleanup(codebook["oil"], table)

[('oil', 1.0), ('aisle-8', 0.03), ('aisle-7', 0.016), ('VISITED', 0.013)]

## One edge, one hypervector

An edge has three parts: a source $s$, a relationship type $r$ and a target $t$. We bind all
three into one hypervector, and permute the target so the edge keeps its direction
(notebook 2's last section):

$$\mathbf{e} = \mathbf{s} \otimes \mathbf{r} \otimes \rho(\mathbf{t})$$

For example, R04 → aisle-9 becomes $\mathbf{R04} \otimes \mathbf{VISITED} \otimes \rho(\mathbf{aisle9})$.

In [6]:
def edge(s, r, t, cb):
    return torchhd.bind(torchhd.bind(cb[s], cb[r]), torchhd.permute(cb[t], shifts=1))

e = edge("R04", "VISITED", "aisle-9", codebook)
print(f"cos(e, R04) = {torchhd.cosine_similarity(e, codebook['R04']).item():+.3f}   (binding: unlike its inputs)")

cos(e, R04) = -0.005   (binding: unlike its inputs)


## The whole graph, one hypervector

Bundle every edge in the graph:

$$\mathbf{G} = \bigoplus_{(s,\, r,\, t)\, \in\, E} \mathbf{s} \otimes \mathbf{r} \otimes \rho(\mathbf{t})$$

`EDGES` has nine edges, so $\mathbf{G}$ is the bundle of nine edge hypervectors. It's still
just $D$ numbers long, whatever the number of edges.

In [7]:
G = torchhd.multiset(torch.stack([edge(s, r, t, codebook) for r, pairs in EDGES.items() for s, t in pairs]))
print("G has", G.numel(), "entries; the first ten are", G[:10].tolist())

G has 10000 entries; the first ten are [1.0, 5.0, 3.0, 1.0, 3.0, 1.0, -3.0, -1.0, 5.0, -7.0]


We store $\mathbf{G}$ in LanceDB too. The whole graph fits in **one row** of a
`graph_memory` table.

In [8]:
memory = db.create_table("graph_memory", [{"name": "warehouse_trial", "vector": G.tolist()}], mode="overwrite")
print(memory.count_rows(), "row,", D * 4 // 1000, "KB of float32")

1 row, 40 KB of float32


## Asking a question

"Where did R04 go?" We know the source and the relationship type, and we want the target.
Undo the edge encoding in reverse order: bind $\mathbf{G}$ with the known parts, then
shift back.

$$\rho^{-1}(\mathbf{G} \otimes \mathbf{R04} \otimes \mathbf{VISITED}) \approx \mathbf{aisle8} \oplus \mathbf{aisle9} \oplus \text{noise}$$

The two edges that start with R04 and VISITED turn back into their targets. The other seven
edges turn into noise: hypervectors that resemble nothing in the codebook.

In [9]:
query = torchhd.permute(torchhd.bind(G, torchhd.bind(codebook["R04"], codebook["VISITED"])), shifts=-1)
top = cleanup(query, table)
top

[('aisle-8', 0.345), ('aisle-9', 0.343), ('oil', 0.019), ('slow_down', 0.017)]

Both aisles score about $1/3$. That's the bundling formula from notebook 2: the query is a
bundle of nine terms (two real answers plus seven noise terms), so each real answer scores
about $1/\sqrt{9}$.

In [10]:
assert {name for name, _ in top[:2]} == {"aisle-8", "aisle-9"}, top

## The same hypervector answers the reverse question

"Who visited aisle-9?" No new index is needed. Bind $\mathbf{G}$ with the relationship
type and the permuted target, and the source comes out:

$$\mathbf{G} \otimes \mathbf{VISITED} \otimes \rho(\mathbf{aisle9}) \approx \mathbf{R04} \oplus \mathbf{R11} \oplus \text{noise}$$

In a graph database you would write a different query, or follow a reverse index. Here it's
the same hypervector and the same three operators.

In [11]:
who = torchhd.bind(G, torchhd.bind(codebook["VISITED"], torchhd.permute(codebook["aisle-9"], shifts=1)))
cleanup(who, table)

[('R04', 0.343), ('R11', 0.328), ('aisle-7', 0.03), ('reroute', 0.013)]

## Two hops, HDC only

"R11 is heading into aisle-9. What should it expect?" We chain two questions. First,
unbind `aisle-9` and `HAS_HAZARD` to find the hazard. Then take the best match and unbind it
with `HANDLED_BY` to find the response.

In [12]:
hop1 = torchhd.permute(torchhd.bind(G, torchhd.bind(codebook["aisle-9"], codebook["HAS_HAZARD"])), shifts=-1)
hazard = cleanup(hop1, table)[0][0]

hop2 = torchhd.permute(torchhd.bind(G, torchhd.bind(codebook[hazard], codebook["HANDLED_BY"])), shifts=-1)
response = cleanup(hop2, table)[0][0]

print(f"aisle-9 → {hazard} → {response}")

aisle-9 → oil → slow_down


Each hop is a **proposal**. The cleanup step always returns *some* nearest name, and nothing
so far has checked that the path aisle-9 → oil → slow_down really exists.

## Validate with the graph

The graph from notebook 1 can check it exactly. We fill the proposed IDs into the
multi-hop Cypher pattern. A non-empty result means the graph confirms the path. We print the
path the graph returned in graph form: nodes in round brackets with their label, and edges
as arrows with their relationship type.

In [13]:
# An f-string is fine here, where the IDs come from our own codebook. Never do this with user input.
q = f'''
MATCH (a:Aisle {{id: 'aisle-9'}})-[:HAS_HAZARD]->(h:Hazard {{id: '{hazard}'}})-[:HANDLED_BY]->(x:Response {{id: '{response}'}})
RETURN a.id, h.id, x.id
'''
confirmed = CypherQuery(q).with_config(cfg).execute(datasets)
assert confirmed.num_rows == 1

a, h, x = confirmed.to_pylist()[0].values()  # the IDs of the three nodes on the path
print(f'''
  ({a} :Aisle)
      │
      │ :HAS_HAZARD
      ▼
  ({h} :Hazard)
      │
      │ :HANDLED_BY
      ▼
  ({x} :Response)

✓ confirmed: this path exists in the graph''')


  (aisle-9 :Aisle)
      │
      │ :HAS_HAZARD
      ▼
  (oil :Hazard)
      │
      │ :HANDLED_BY
      ▼
  (slow_down :Response)

✓ confirmed: this path exists in the graph


## When HDC guesses wrong

At $D = 10{,}000$ the noise is tiny, about $1/\sqrt{D} = 0.01$, and the answers are clean. A
small $D$ makes the noise much larger. Let's rebuild everything at $D = 64$, where the noise
is about $1/\sqrt{64} = 0.125$, close to the $1/3$ signal of a real answer.

In [14]:
torch.manual_seed(0)
D_small = 64

small_codebook = dict(zip(names, torchhd.random(len(names), D_small, vsa="MAP")))
G_small = torchhd.multiset(torch.stack([edge(s, r, t, small_codebook) for r, pairs in EDGES.items() for s, t in pairs]))
small_table = db.create_table(
    "hypervectors_d64",
    [{"name": n, "kind": kinds[n], "vector": small_codebook[n].tolist()} for n in names],
    mode="overwrite",
)

Ask "Where did R04 go?" again, and keep every candidate that scores above a threshold of
$0.05$. With seed 0 and $D = 64$, some names that aren't aisles R04 visited get past the
threshold too.

In [15]:
THRESHOLD = 0.05
query_small = torchhd.permute(torchhd.bind(G_small, torchhd.bind(small_codebook["R04"], small_codebook["VISITED"])), shifts=-1)
candidates = [(name, score) for name, score in cleanup(query_small, small_table, k=len(names)) if score > THRESHOLD]
candidates

[('aisle-9', 0.378),
 ('aisle-8', 0.286),
 ('HAS_HAZARD', 0.245),
 ('R04', 0.122),
 ('forklift', 0.122),
 ('aisle-7', 0.082),
 ('VISITED', 0.071)]

Now the graph checks each candidate. Only real `VISITED` edges from R04 to an `Aisle` node
come back non-empty, so wrong guesses, including ones that aren't even aisles, get rejected.

In [16]:
print(f"{'candidate':12} {'HDC score':>9}   graph")
verdicts = {}
for name, score in candidates:
    q = f"MATCH (r:Robot {{id: 'R04'}})-[:VISITED]->(a:Aisle {{id: '{name}'}}) RETURN a.id"
    verdicts[name] = CypherQuery(q).with_config(cfg).execute(datasets).num_rows > 0
    print(f"{name:12} {score:9.3f}   {'confirmed' if verdicts[name] else 'rejected'}")

candidate    HDC score   graph
aisle-9          0.378   confirmed
aisle-8          0.286   confirmed
HAS_HAZARD       0.245   rejected
R04              0.122   rejected
forklift         0.122   rejected
aisle-7          0.082   rejected
VISITED          0.071   rejected


In [17]:
assert verdicts.get("aisle-8") and verdicts.get("aisle-9"), verdicts
assert not all(verdicts.values()), "expected at least one false candidate at D = 64"

## Wrap-up

| | HDC | Graph |
|---|---|---|
| Storage | one fixed-size hypervector, here a single row | one table per label and per relationship type |
| A query is | a few multiplications and one nearest-neighbour search | a pattern compiled into joins |
| Answers are | approximate **proposals**, scored | exact, auditable **confirmations** |
| Can answer | questions nobody wrote a query for, in either direction | only the patterns you write |

HDC gives fast, approximate, compositional proposals from one fixed-size hypervector. The
graph gives exact confirmation of each one. Hopefully, these notebooks give you a good feel for the HDC + graph paradigm
we showcase in the talk!

> **HDC proposes candidates. The graph validates whether those paths actually exist.**